[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/Latent/blob/main/notebooks/colab/Stage_1_Data_Collection.ipynb)

---


In [ ]:
# ══════════════════════════════════════════
# Google Colab Setup
# ══════════════════════════════════════════
# Run this cell first to set up the environment in Google Colab.

import os
import sys

# Check if running in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Mount Google Drive for data persistence
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Set project root on Google Drive
    PROJECT_ROOT = '/content/drive/MyDrive/Latent'
    os.makedirs(PROJECT_ROOT, exist_ok=True)
    os.makedirs(f'{PROJECT_ROOT}/intermediate_data', exist_ok=True)
    os.makedirs(f'{PROJECT_ROOT}/utilities', exist_ok=True)
    
    # Clone or update the repo
    if not os.path.exists('/content/Latent'):
        !git clone https://github.com/YOUR_USERNAME/Latent.git /content/Latent
    
    # Install dependencies
    !pip install -q spacy nltk ftfy emoji langdetect sentence-transformers rank-bm25 tqdm
    !python -m spacy download en_core_web_sm
    
    # Set working directory
    os.chdir('/content/Latent')
    print(f'Working directory: {os.getcwd()}')
    print(f'Data directory: {PROJECT_ROOT}')
else:
    PROJECT_ROOT = '.'
    print('Running locally. Ensure dependencies are installed via: pip install -r requirements.txt')


# Stage 1: Data Collection and Data Cleaning

### Project Structure

```bash
Latent/
    │
    ├─ data/
    │   ├─ inverted_index/
    │   │   ├─ inverted_index_tfidf_titles.json
    │   │   └─ inverted_index_tfidf_fulltext.json
    │   │
    │   ├─ models/
    │   │   ├─ sentence_bert_model/
    │   │   ├─ bm25_comments_model.joblib
    │   │   ├─ bm25_fulltext_model.joblib
    │   │   ├─ bm25_titles_model.joblib
    │   │   ├─ tfidf_comments_vectorizer.joblib
    │   │   └─ tfidf_posts_vectorizer.joblib
    │   │
    │   ├─ vector_database/
    │   │   ├─ bert_comments_embeddings.npy
    │   │   ├─ bert_contents_embeddings.npy
    │   │   ├─ bert_fulltext_embeddings.npy
    │   │   ├─ bert_titles_embeddings.npy
    │   │   ├─ bm25_comments.npz
    │   │   ├─ bm25_contents.npz
    │   │   ├─ bm25_fulltext.npz
    │   │   ├─ bm25_titles.npz
    │   │   ├─ tfidf_comments.npz
    │   │   ├─ tfidf_contents.npz
    │   │   ├─ tfidf_fulltext.npz
    │   │   └─ tfidf_titles.npz
    │   │
    │   ├─ PostVault.csv
    │   ├─ CommentVault.csv
    │   └─ raw_data/
    │
    ├─ sentiment_plots/
    │   ├─ emotion_plots/
    │   ├─ emotion_dashboard.py
    │   └─ plot_emotion_summary.py
    │
    ├─ Stage_0_Introduction.ipynb                               
-----------------------------------------------------------------            
│   ├─ Stage_1_Data_Collection_and_Data_Cleaning.ipynb          │
-----------------------------------------------------------------
    ├─ Stage_2_POS_and_NER_Tagging.ipynb
    ├─ Stage_3_Singlish_Normalisation.ipynb
    ├─ Stage_4_Singlish_to_English_Conversion.ipynb     
    ├─ Stage_5_Common_Normalisation.ipynb
    ├─ Stage_6_Vector_Space_Model_and_Inverted_Index.ipynb
    ├─ Stage_7_Sentiment_Analysis.ipynb
    ├─ Stage_8_Clustering_and_Visualization.ipynb       
    ├─ Stage_9_Document_Search.ipynb
    └─ utilities/
        │
        ├─ pp_class.py
        ├─ singlish_dictionary.json
        ├─ singlish_regex_to_text.txt
        └─ slang_dictionary.csv
```

## To access the mounted folder directly.

### Step 1: Add the Shared Folder to your Google Drive (if you haven't already)

1.  **Open Google Drive** (drive.google.com) in your web browser.
2.  Go to the **'Shared with me'** section.
3.  Locate the folder that was shared with you.
4.  **Right-click** on the shared folder.
5.  Select **'Add shortcut to Drive'** (or 'Add to My Drive', depending on the interface). Choose a location within 'My Drive' if prompted, or simply add it to the root of 'My Drive'.

This creates a shortcut to the shared folder in your 'My Drive', making it accessible when Colab mounts your personal Drive.
### Step 2: Access the Shared Folder in Colab (No action needed)

Once your Drive is mounted, you can navigate to the shared folder. If you added a shortcut, it will appear in your 'My Drive' just like any other folder. The path will typically be `/content/gdrive/MyDrive/YourSharedFolderName`.

It should be able to run the script below already

In [ ]:
# -----------
# Mount Drive
# -----------
from google.colab import drive
drive.mount('/content/drive')

## Dependencies

### Install Dependencies

In [ ]:
!pip install emoji ftfy langdetect import-ipynb contractions

### Change Directory

In [ ]:
import os

# Path
# ----

FOLDER_PATH: str = "/content/drive/MyDrive/Latent/"

os.chdir(FOLDER_PATH)

### Import Essential Library

In [ ]:
import import_ipynb
import pandas as pd
from typing import Any
from Stage_0_Introduction import display_first_text_column_head, save_dfs_to_csv, output_base_dir, save_single_df_to_csv

## Reddit Preprocessor Class

In [ ]:
from __future__ import annotations

import glob
import html
import logging
import re
from typing import Any

import contractions
import emoji
import ftfy
import pandas as pd
from langdetect import LangDetectException, detect

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)


class RedditPreprocessor:
    """
    Single-class Reddit preprocessing pipeline for posts and comments.
    """

    REMOVED_MARKERS = {"[removed]", "[deleted]", "[deleted by user]"}
    KNOWN_BOTS = {
        "AutoModerator",
        "SG_wormsbot",
        "RepostSleuthBot",
        "RemindMeBot",
        "reddit_repost_sleuth",
        "TotesMessenger",
        "converter-bot",
    }
    SCORE_BINS = [-float("inf"), 0, 5, 25, 100, float("inf")]
    SCORE_LABELS = ["negative", "low", "medium", "high", "viral"]

    POSTS_CANONICAL_COLS = [
        "id",
        "title",
        "selftext",
        "author",
        "score",
        "upvote_ratio",
        "num_comments",
        "created_utc",
        "edited",
        "distinguished",
        "stickied",
        "over_18",
        "spoiler",
        "locked",
        "archived",
        "is_original_content",
        "is_self",
        "permalink",
        "url",
        "domain",
        "subreddit",
        "subreddit_id",
        "link_flair_text",
        "link_flair_css_class",
        "author_flair_text",
        "author_flair_css_class",
        "gilded",
        "total_awards_received",
        "is_video",
        "media",
        "thumbnail",
        "post_hint",
        "treatment_tags",
        "secure_media",
        "preview",
        "downloaded_media",
    ]

    COMMENTS_CANONICAL_COLS = [
        "id",
        "parent_id",
        "post_id",
        "body",
        "author",
        "score",
        "created_utc",
        "edited",
        "distinguished",
        "stickied",
        "is_submitter",
        "author_flair_text",
        "author_flair_css_class",
        "gilded",
        "total_awards_received",
        "permalink",
        "subreddit",
        "subreddit_id",
        "depth",
        "controversiality",
        "collapsed",
        "collapsed_reason",
        "locked",
    ]

    _RE_URL = re.compile(r"https?://\S+|www\.\S+")
    _RE_MENTIONS = re.compile(r"\b[ur]/\w+")
    _RE_MD_LINK = re.compile(r"\[([^\]]+)\]\([^)]+\)")
    _RE_MD_IMAGE = re.compile(r"!\[[^\]]*\]\([^)]+\)")
    _RE_BOLD_ITALIC = re.compile(r"\*{1,3}(.+?)\*{1,3}")
    _RE_STRIKETHROUGH = re.compile(r"~~(.+?)~~")
    _RE_INLINE_CODE = re.compile(r"`[^`]+`")
    _RE_CODE_BLOCK = re.compile(r"```[\s\S]*?```")
    _RE_BLOCKQUOTE = re.compile(r"^>+\s?", re.MULTILINE)
    _RE_HEADER = re.compile(r"^#{1,6}\s+", re.MULTILINE)
    _RE_HR = re.compile(r"^[-*_]{3,}\s*$", re.MULTILINE)
    _RE_MULTI_UNDERSCORE = re.compile(r"_+")
    _RE_ELONGATION       = re.compile(r"(.)\1{2,}")
    _RE_REPEATED_PUNCT   = re.compile(r"([!?.])\1+")
    _RE_NON_ALPHA        = re.compile(r"[^a-z0-9\s]")

    # ASCII emoticons — matched between non-word boundaries or string edges.
    # Order matters: longer/more specific patterns first.
    _ASCII_EMOJI_MAP: list[tuple[re.Pattern, str]] = [
        # crying / very sad
        (re.compile(r"(?<!\w)[;:]['`]-?\((?!\w)"),        "crying"),
        (re.compile(r"(?<!\w)t[_.]t(?<!\w)"),              "crying"),
        (re.compile(r"(?<!\w)q[_.]q(?<!\w)"),              "crying"),
        # angry
        (re.compile(r"(?<!\w)>:-\((?!\w)"),              "angry"),
        (re.compile(r"(?<!\w)>_<(?<!\w)"),                 "frustrated"),
        # surprised / shocked
        (re.compile(r"(?<!\w):-?[oO](?!\w)"),             "surprised"),
        (re.compile(r"(?<!\w)[oO][_.-][oO](?!\w)"),       "shocked"),
        (re.compile(r"(?<!\w)[oO][_.-][oO](?!\w)"),       "shocked"),
        # grinning / very happy
        (re.compile(r"(?<!\w)x[Dd](?!\w)"),               "grinning"),
        (re.compile(r"(?<!\w):-?[dD](?!\w)"),             "grinning"),
        (re.compile(r"(?<!\w)=\s?[dD](?!\w)"),            "grinning"),
        (re.compile(r"(?<!\w)\^[_.-]?\^(?!\w)"),         "happy"),
        # winking
        (re.compile(r"(?<!\w);-?\)(?!\w)"),               "winking"),
        (re.compile(r"(?<!\w);-?[dD](?!\w)"),             "winking grin"),
        # happy / smile
        (re.compile(r"(?<!\w)[=:]-?\)(?!\w)"),            "happy"),
        (re.compile(r"(?<!\w>[=:])\)(?!\w)"),             "happy"),
        # sad / frown
        (re.compile(r"(?<!\w)[=:]-?\((?!\w)"),            "sad"),
        (re.compile(r"(?<!\w>[=:])\((?!\w)"),             "sad"),
        # skeptical / unsure
        (re.compile(r"(?<!\w)[=:]-?[/\\](?<!\w)"),         "skeptical"),
        # tongue / playful
        (re.compile(r"(?<!\w)[=:]-?[pP](?!\w)"),         "playful"),
        # neutral
        (re.compile(r"(?<!\w)[=:]-?\|(?<!\w)"),            "neutral"),
        # love / heartbreak
        (re.compile(r"(?<!\w)<3(?<!\w)"),                  "love"),
        (re.compile(r"(?<!\w)</3(?<!\w)"),                 "heartbreak"),
    ]

    def __init__(self, min_words: int = 3, lang_detect: bool = True) -> None:
        self.min_words = min_words
        self.lang_detect = lang_detect

    def harmonise_schema(self, df: pd.DataFrame, canonical_cols: list[str]) -> pd.DataFrame:
        out = df.copy()
        for col in canonical_cols:
            if col not in out.columns:
                out[col] = None
        return out[canonical_cols]

    def dedup(self, df: pd.DataFrame, id_col: str = "id") -> pd.DataFrame:
        before = len(df)
        out = df.drop_duplicates(subset=[id_col])
        log.info("Dedup: %d -> %d rows (removed %d)", before, len(out), before - len(out))
        return out

    def drop_removed(self, df: pd.DataFrame, text_col: str) -> pd.DataFrame:
        before = len(df)
        mask = df[text_col].astype(str).str.strip().isin(self.REMOVED_MARKERS)
        out = df[~mask]
        log.info("Drop removed: %d -> %d rows (removed %d)", before, len(out), before - len(out))
        return out

    def drop_bots_and_mods(self, df: pd.DataFrame) -> pd.DataFrame:
        before = len(df)
        is_bot = df["author"].isin(self.KNOWN_BOTS)
        is_mod = df["distinguished"].astype(str).str.lower().isin({"moderator", "admin"})
        is_stickied = df.get("stickied", pd.Series(False, index=df.index)).astype(bool)
        out = df[~(is_bot | is_mod | is_stickied)]
        log.info("Drop bots/mods/stickied: %d -> %d rows (removed %d)", before, len(out), before - len(out))
        return out

    def drop_nsfw(self, df: pd.DataFrame) -> pd.DataFrame:
        before = len(df)
        is_nsfw = df["over_18"].astype(str).str.lower() == "true"
        out = df[~is_nsfw]
        log.info("Drop NSFW: %d -> %d rows (removed %d)", before, len(out), before - len(out))
        return out

    def strip_markdown(self, text: str) -> str:
        text = html.unescape(text)
        text = self._RE_CODE_BLOCK.sub(" ", text)
        text = self._RE_MD_IMAGE.sub(" ", text)
        text = self._RE_MD_LINK.sub(r"\1", text)
        text = self._RE_URL.sub(" ", text)
        text = self._RE_MENTIONS.sub(" ", text)
        text = self._RE_BOLD_ITALIC.sub(r"\1", text)
        text = self._RE_STRIKETHROUGH.sub(r"\1", text)
        text = self._RE_INLINE_CODE.sub(" ", text)
        text = self._RE_BLOCKQUOTE.sub(" ", text)
        text = self._RE_HEADER.sub(" ", text)
        text = self._RE_HR.sub(" ", text)
        return text

    @staticmethod
    def fix_encoding(text: str) -> str:
        return ftfy.fix_text(text)

    @staticmethod
    def normalise_text(text: str) -> str:
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()

    def resolve_ascii_emojis(self, text: str) -> str:
        """
        Replace ASCII emoticons with English labels (e.g. ':)' -> 'happy').
        """
        for pattern, label in self._ASCII_EMOJI_MAP:
            text = pattern.sub(f" {label} ", text)
        return text

    def normalise_for_classification(self, text: str) -> str:
        """
        Lowercase, expand contractions, resolve ASCII emojis, collapse elongation, strip non-alphanumeric.
        """
        text = text.lower()
        text = contractions.fix(text)                       # don't -> do not
        text = self.resolve_ascii_emojis(text)              # :) -> happy
        text = self._RE_ELONGATION.sub(r"\1\1", text)      # coooool -> cool
        text = self._RE_REPEATED_PUNCT.sub(r"\1", text)    # !!! -> !
        text = self._RE_NON_ALPHA.sub(" ", text)           # keep a-z, 0-9, spaces
        return re.sub(r"\s+", " ", text).strip()

    def drop_short(self, df: pd.DataFrame, text_col: str) -> pd.DataFrame:
        before = len(df)
        word_count = df[text_col].astype(str).str.split().str.len()
        out = df[word_count >= self.min_words]
        log.info(
            "Drop short (<%d words): %d -> %d rows (removed %d)",
            self.min_words,
            before,
            len(out),
            before - len(out),
        )
        return out

    def _emoji_to_english(self, text: str) -> str:
        """
        Translate emojis to short English labels
        """
        demojized = emoji.demojize(str(text), delimiters=(" ", " "))
        demojized = self._RE_MULTI_UNDERSCORE.sub("_", demojized)
        demojized = demojized.replace("_", " ")
        return demojized

    def process_emoji(self, df: pd.DataFrame, text_col: str) -> pd.DataFrame:
        out = df.copy()
        out["has_emoji"] = out[text_col].astype(str).apply(lambda t: bool(emoji.emoji_list(t)))
        out[text_col] = out[text_col].astype(str).apply(self._emoji_to_english).apply(self.normalise_text)
        return out

    @staticmethod
    def detect_language(text: str) -> str:
        try:
            return detect(str(text))
        except LangDetectException:
            return "unknown"

    def extract_features(self, df: pd.DataFrame, is_posts: bool, text_col_for_len_count: str = "cleaned_text") -> pd.DataFrame:
        out = df.copy()
        out["created_utc"] = pd.to_datetime(out["created_utc"], utc=True, errors="coerce")
        out["year"] = out["created_utc"].dt.year
        out["month"] = out["created_utc"].dt.month
        out["day_of_week"] = out["created_utc"].dt.day_name()
        out["hour"] = out["created_utc"].dt.hour
        out["score_bucket"] = pd.cut(out["score"].astype(float), bins=self.SCORE_BINS, labels=self.SCORE_LABELS)
        # Use the specified text_col_for_len_count for length and word count
        out["text_length"] = out[text_col_for_len_count].astype(str).str.len()
        out["word_count"] = out[text_col_for_len_count].astype(str).str.split().str.len()

        if is_posts:
            # Changed 'selftext' to 'content' as it was renamed earlier in the pipeline
            out["has_body"] = out["content"].notna() & (out["content"].astype(str).str.strip() != "")
        else:
            depth = out["depth"].fillna(0).astype(float)
            out["depth_bucket"] = pd.cut(
                depth, bins=[-1, 0, 2, 5, float("inf")], labels=["top_level", "shallow", "mid", "deep"]
            )
        return out

In [ ]:
reddit = RedditPreprocessor(min_words=3, lang_detect=False)

## Data Loading

In [ ]:
import os

# Define the path to your specific folder

folder_path = '/content/drive/MyDrive/Latent'
data_folder_path = '/content/drive/MyDrive/Latent/data/raw_data'

# Check if the folder exists and list its contents
if os.path.exists(folder_path):
    print(f"Contents of '{folder_path}':")
    for item in os.listdir(folder_path):
        print(item)
else:
    print(f"The folder '{folder_path}' does not exist. Please ensure the path is correct and the drive is mounted.")

In [ ]:
# Initialize an empty list to store the individual DataFrames
singapore_dfs = []

# Iterate through the files in the specified directory
for item in os.listdir(data_folder_path):
    file_path = os.path.join(data_folder_path, item)
    print(f"Attempting to read: {item}")
    df = pd.read_csv(file_path, encoding='utf-8',  on_bad_lines="skip",)
    try:
      singapore_dfs.append(df)
    except Exception as e:
      print(f"Error reading {item}: {e}")
      continue

    print(f"Successfully loaded '{item}' with UTF-8 encoding.")

print(f"\nTotal {len(singapore_dfs)} relevant DataFrame(s) loaded.")
if singapore_dfs:
    print("First few rows of the first loaded DataFrame:")
    display(singapore_dfs[0].head())

## Social Media Text Cleaning

Tasks:
1. Removing URLs
1. Stripping user mentions
1. Reducing character elongation
1. Normalizing punctuation
1. De-deduplicate
1. Remove bots and mods
1. Remove deleted posts
1. Remove 3 word posts
1. Expand contractions
1. Convert emoji to text

In [ ]:
#  Resolve ASCII emojis and expand contractions BEFORE clean_text runs
_PREPROCESS_COLS = ['title', 'selftext', 'body']
for df in singapore_dfs:
    for col in _PREPROCESS_COLS:
        if col in df.columns:
            # Only apply emoji-to-English conversion at this stage to raw columns.
            # Further normalization (contractions, ASCII emojis, lowercasing, etc.)
            # will be handled by reddit.normalise_for_classification later.
            df[col] = df[col].apply(
                lambda x: reddit._emoji_to_english(str(x)) if pd.notna(x) else x
            )

# Split the list of DataFrames into posts and comments based on column presence
post_dfs    = [df for df in singapore_dfs if 'title' in df.columns]
comment_dfs = [df for df in singapore_dfs if 'body'  in df.columns and 'title' not in df.columns]

# Concatenate the lists of DataFrames into single DataFrames for posts and comments
df_posts    = pd.concat(post_dfs,    ignore_index=True) if post_dfs    else pd.DataFrame()
df_comments = pd.concat(comment_dfs, ignore_index=True) if comment_dfs else pd.DataFrame()

# Define columns to keep for posts and comments explicitly
POST_COLS_TO_KEEP = [
    "id", "title", "content", # Keep original raw columns, 'selftext' renamed to 'content'
    "cleaned_title", "cleaned_content", # New cleaned columns
    "score", "upvote_ratio", "num_comments", "created_utc",
    "subreddit_id", "has_emoji", "year", "month", "day_of_week",
    "hour", "score_bucket", "text_length", "word_count"
]

COMMENT_COLS_TO_KEEP = [
    "id", "parent_id", "post_id", "text", "score", "created_utc", # 'body' renamed to 'content'
    "subreddit_id", "depth", "controversiality", "cleaned_text", "has_emoji",
    "year", "month", "day_of_week", "hour", "score_bucket", "text_length",
    "word_count", "depth_bucket"
]

# Process Posts DataFrame
if not df_posts.empty:
    print(f"Initial df_posts size: {len(df_posts)} rows")
    # Harmonize schema to ensure all expected columns are present
    df_posts = reddit.harmonise_schema(df_posts, reddit.POSTS_CANONICAL_COLS)
    print(f"df_posts after harmonise_schema: {len(df_posts)} rows")

    # Drop rows where 'id' is NaN
    before_drop_id = len(df_posts)
    df_posts.dropna(subset=['id'], inplace=True)
    print(f"df_posts after dropping NaN IDs: {len(df_posts)} (removed {before_drop_id - len(df_posts)}) rows")

    # Remove duplicate posts based on 'id'
    df_posts = reddit.dedup(df_posts, 'id')
    print(f"df_posts after dedup: {len(df_posts)} rows")
    # Drop posts that have been marked as 'removed' or 'deleted'
    df_posts = reddit.drop_removed(df_posts, 'title')
    print(f"df_posts after drop_removed (title): {len(df_posts)} rows")
    # Remove posts from known bots or moderators
    df_posts = reddit.drop_bots_and_mods(df_posts)
    print(f"df_posts after drop_bots_and_mods: {len(df_posts)} rows")

    # Rename 'selftext' to 'content' earlier in the pipeline
    df_posts = df_posts.rename(columns={"selftext": "content"})

    # Drop rows where both 'title' and 'content' are NaN (representing empty posts)
    before_drop_empty_post = len(df_posts)
    df_posts.dropna(subset=['title', 'content'], how='all', inplace=True)
    print(f"df_posts after dropping empty posts: {len(df_posts)} (removed {before_drop_empty_post - len(df_posts)}) rows")

    # Apply normalization for classification to 'title' and 'content' separately
    df_posts['cleaned_title'] = df_posts['title'].fillna('').str.strip().apply(reddit.normalise_for_classification)
    df_posts['cleaned_content'] = df_posts['content'].fillna('').str.strip().apply(reddit.normalise_for_classification)

    # Create a temporary combined cleaned text for post-level operations (like emoji detection, short-post filtering, and feature extraction)
    df_posts['temp_cleaned_text_for_post_ops'] = (
        df_posts['cleaned_title'].fillna('') + ' ' + df_posts['cleaned_content'].fillna('')
    ).str.strip()

    # Process emojis (adds 'has_emoji' and applies _emoji_to_english and normalise_text to the temporary combined text)
    df_posts = reddit.process_emoji(df_posts, 'temp_cleaned_text_for_post_ops')
    print(f"df_posts after process_emoji: {len(df_posts)} rows")
    # Remove posts with fewer words than the defined minimum (based on combined temporary text)
    df_posts = reddit.drop_short(df_posts, 'temp_cleaned_text_for_post_ops')
    print(f"df_posts after drop_short: {len(df_posts)} rows")
    # Extract additional features like time, score buckets, text length, and word count (uses temporary combined text)
    df_posts = reddit.extract_features(df_posts, is_posts=True, text_col_for_len_count='temp_cleaned_text_for_post_ops')
    print(f"df_posts after extract_features: {len(df_posts)} rows")

    # Drop the temporary combined text column as it's not part of the final output requirements
    df_posts.drop(columns=['temp_cleaned_text_for_post_ops'], inplace=True, errors='ignore')

    # Ensure 'title' and 'content' are empty strings if NaN, before final selection
    if 'title' in df_posts.columns:
        df_posts['title'] = df_posts['title'].fillna('')

    if 'cleaned_title' in df_posts.columns:
        df_posts['cleaned_title'] = df_posts['cleaned_title'].fillna('')


    if 'content' in df_posts.columns:
        df_posts['content'] = df_posts['content'].fillna('')

    if 'cleaned_content' in df_posts.columns:
        df_posts['cleaned_content'] = df_posts['cleaned_content'].fillna('')


    # Fill NaNs for specific columns before final selection
    numeric_cols_posts = ["score", "upvote_ratio", "num_comments", "text_length", "word_count", "year", "month", "hour"]
    categorical_cols_posts = ["day_of_week", "score_bucket", "subreddit_id"]
    boolean_cols_posts = ["has_emoji"]

    for col in numeric_cols_posts:
        if col in df_posts.columns:
            df_posts[col] = df_posts[col].fillna(0)
    for col in categorical_cols_posts:
        if col in df_posts.columns:
            
            if isinstance(df_posts[col].dtype, pd.CategoricalDtype):
                df_posts[col] = df_posts[col].astype(object)
            df_posts[col] = df_posts[col].fillna('unknown')
    for col in boolean_cols_posts:
        if col in df_posts.columns:
            df_posts[col] = df_posts[col].fillna(False)

    # Select only the desired columns
    df_posts = df_posts[POST_COLS_TO_KEEP]
    df_posts['fulltext'] = df_posts['title'] + ' ' + df_posts['content']
    print(f"Posts: {len(df_posts)} rows")

# Process Comments DataFrame
if not df_comments.empty:
    print(f"\nInitial df_comments size: {len(df_comments)} rows")
    # Harmonize schema to ensure all expected columns are present
    df_comments = reddit.harmonise_schema(df_comments, reddit.COMMENTS_CANONICAL_COLS)
    print(f"df_comments after harmonise_schema: {len(df_comments)} rows")

    # Drop rows where 'id' is NaN
    before_drop_id = len(df_comments)
    df_comments.dropna(subset=['id'], inplace=True)
    print(f"df_comments after dropping NaN IDs: {len(df_comments)} (removed {before_drop_id - len(df_comments)}) rows")

    # Remove duplicate comments based on 'id'
    df_comments = reddit.dedup(df_comments, 'id')
    print(f"df_comments after dedup: {len(df_comments)} rows")
    # Drop comments that have been marked as 'removed' or 'deleted'
    df_comments = reddit.drop_removed(df_comments, 'body')
    print(f"df_comments after drop_removed (body): {len(df_comments)} rows")
    # Remove comments from known bots or moderators
    df_comments = reddit.drop_bots_and_mods(df_comments)
    print(f"df_comments after drop_bots_and_mods: {len(df_comments)} rows")

    # Rename 'body' to 'text' earlier in the pipeline
    df_comments = df_comments.rename(columns={"body": "text"})

    # Drop rows where 'text' is NaN (representing empty comments)
    before_drop_empty_comment = len(df_comments)
    df_comments.dropna(subset=['text'], inplace=True)
    print(f"df_comments after dropping empty comments: {len(df_comments)} (removed {before_drop_empty_comment - len(df_comments)}) rows")

    # Create a clean parent_id (used for some merges or identification)
    df_comments['parent_id'] = df_comments['parent_id'].str.replace('t3_', '')

    # Apply normalization for classification to the pre-cleaned 'text' column
    df_comments['cleaned_text'] = (
        df_comments['text'].fillna('').str.strip().apply(reddit.normalise_for_classification)
    )
    # Process emojis again to ensure consistency
    df_comments = reddit.process_emoji(df_comments, 'cleaned_text')
    print(f"df_comments after process_emoji: {len(df_comments)} rows")
    # Remove comments with fewer words than the defined minimum
    df_comments = reddit.drop_short(df_comments, 'cleaned_text')
    print(f"df_comments after drop_short: {len(df_comments)} rows")
    # Extract additional features like time, score buckets, and text length/word count
    df_comments = reddit.extract_features(df_comments, is_posts=False, text_col_for_len_count='cleaned_text')
    print(f"df_comments after extract_features: {len(df_comments)} rows")

    # Ensure 'content' and 'cleaned_text' are empty strings if NaN, before final selection
    if 'content' in df_comments.columns:
        df_comments['content'] = df_comments['content'].fillna('')

    if 'cleaned_text' in df_comments.columns:
        df_comments['cleaned_text'] = df_comments['cleaned_text'].fillna('')

    # Fill NaNs for specific columns before final selection
    numeric_cols_comments = ["score", "depth", "controversiality", "text_length", "word_count", "year", "month", "hour"]
    categorical_cols_comments = ["day_of_week", "score_bucket", "depth_bucket", "subreddit_id"]
    boolean_cols_comments = ["has_emoji"]

    for col in numeric_cols_comments:
        if col in df_comments.columns:
            df_comments[col] = df_comments[col].fillna(0)
    for col in categorical_cols_comments:
        if col in df_comments.columns:
            if isinstance(df_comments[col].dtype, pd.CategoricalDtype):
                df_comments[col] = df_comments[col].astype(object)
            df_comments[col] = df_comments[col].fillna('unknown')
    for col in boolean_cols_comments:
        if col in df_comments.columns:
            df_comments[col] = df_comments[col].fillna(False)

    # Select only the desired columns
    df_comments = df_comments[COMMENT_COLS_TO_KEEP]

    print(f"Comments: {len(df_comments)} rows")

In [ ]:
comments = df_comments.copy()
print("-------------------------------------------------")
print("First few rows of comments['cleaned_text'] column")
print("-------------------------------------------------")
display(comments[['text', 'cleaned_text']].head())
print()

posts = df_posts.copy()
print("-----------------------------------------------------------------------------")
print("First few rows of posts['cleaned_title'] and posts['cleaned_content'] columns")
print("-----------------------------------------------------------------------------")
display(posts[['title', 'content', 'cleaned_title', 'cleaned_content']].head())

### Load Files

In [ ]:
df_posts = pd.read_csv("/content/drive/MyDrive/Latent/data/PostVault.csv", keep_default_na=False)
df_comments = pd.read_csv("/content/drive/MyDrive/Latent/data/CommentVault.csv", keep_default_na=False)

### Save Files

In [ ]:
# ---------
# PostVault
# ---------
df_posts = df_posts.reindex(posts.index)
for col in posts.columns:
    df_posts.loc[:, col] = posts[col]

display(df_posts.head())
print("-----------")
print("Posts Shape")
print("-----------")
print(df_posts.shape)
save_single_df_to_csv(df_posts, "/content/drive/MyDrive/Latent/data/", "PostVault")

# ------------
# CommentVault
# ------------
df_comments = df_comments.reindex(comments.index)
for col in comments.columns:
    df_comments.loc[:, col] = comments[col]

display(df_comments.head())
print("--------------")
print("Comments Shape")
print("--------------")
print(df_comments.shape)
save_single_df_to_csv(df_comments, "/content/drive/MyDrive/Latent/data/", "CommentVault")